# 02 - Order Risk Scoring Model
## XGBoost Classifier for Delivery Success Prediction

This notebook trains an XGBoost model to predict whether an order will be
successfully delivered or will fail (returned/cancelled/no_answer).

### Target Variable
- `is_delivered = 1` → Order delivered successfully
- `is_delivered = 0` → Order failed (cancelled, returned, etc.)

### Output
- Trained model saved to `trained_models/risk_model.joblib`
- Feature engineer saved to `trained_models/feature_engineer.joblib`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, accuracy_score
)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

sys.path.insert(0, str(Path('.').resolve().parent))
from data.mapping import STATE_TO_WILAYA, STATUS_MAPPING, CATEGORY_TRANSLATION, BRL_TO_DZD
from app.models.features import FeatureEngineer

sns.set_style('whitegrid')
MODEL_DIR = Path('../trained_models')
MODEL_DIR.mkdir(exist_ok=True)

print('Libraries loaded!')

## 1. Load & Prepare Data

Option A: Load from prepared CSV (if you ran `prepare_data.py`)
Option B: Load from raw Olist files

In [ ]:
PREPARED_PATH = Path('../data/prepared/crm_orders.csv')
RAW_DIR = Path('../data/olist')

if PREPARED_PATH.exists():
    print('Loading prepared data...')
    df = pd.read_csv(PREPARED_PATH)
    df['order_date'] = pd.to_datetime(df['order_date'])
    print(f'Loaded {len(df)} orders')
else:
    print('Prepared data not found. Loading raw Olist data...')
    print('Run: python -m data.prepare_data   (from ml-service directory)')
    
    # Quick inline preparation
    orders = pd.read_csv(RAW_DIR / 'olist_orders_dataset.csv')
    customers = pd.read_csv(RAW_DIR / 'olist_customers_dataset.csv')
    items = pd.read_csv(RAW_DIR / 'olist_order_items_dataset.csv')
    products = pd.read_csv(RAW_DIR / 'olist_products_dataset.csv')
    payments = pd.read_csv(RAW_DIR / 'olist_order_payments_dataset.csv')
    
    df = orders.merge(customers, on='customer_id', how='left')
    
    items_agg = items.groupby('order_id').agg(
        n_items=('order_item_id', 'count'),
        subtotal=('price', 'sum'),
    ).reset_index()
    
    items_products = items.merge(products[['product_id', 'product_category_name', 'product_weight_g']], on='product_id', how='left')
    primary_cat = items_products.groupby('order_id').agg(
        product_category=('product_category_name', 'first'),
        avg_product_weight=('product_weight_g', lambda x: x.mean() / 1000),
    ).reset_index()
    
    pay_agg = payments.groupby('order_id').agg(payment_value=('payment_value', 'sum')).reset_index()
    
    df = df.merge(items_agg, on='order_id', how='left')
    df = df.merge(primary_cat, on='order_id', how='left')
    df = df.merge(pay_agg, on='order_id', how='left')
    
    df['order_date'] = pd.to_datetime(df['order_purchase_timestamp'])
    df['is_delivered'] = (df['order_status'] == 'delivered').astype(int)
    df['customer_state'] = df['customer_state']
    df['total_amount'] = (df['payment_value'].fillna(0) * BRL_TO_DZD).round(2)
    df['subtotal'] = (df['subtotal'].fillna(0) * BRL_TO_DZD).round(2)
    df['shipping_cost'] = df['customer_state'].map(
        lambda s: {'zone_1': 400, 'zone_2': 600, 'zone_3': 900}.get(
            STATE_TO_WILAYA.get(s, {}).get('zone', 'zone_1'), 400)
    )
    df['discount'] = 0
    df['estimated_delivery_days'] = (
        pd.to_datetime(df['order_estimated_delivery_date']) - df['order_date']
    ).dt.days.clip(lower=1)
    df['customer_phone_2'] = np.where(np.random.random(len(df)) < 0.4, 'yes', None)
    
    # Customer history
    cust_stats = df.groupby('customer_unique_id').agg(
        customer_order_count=('order_id', 'count'),
        customer_success_rate=('is_delivered', 'mean'),
    ).reset_index().rename(columns={'customer_unique_id': 'customer_id_join'})
    df = df.merge(cust_stats, left_on='customer_unique_id', right_on='customer_id_join', how='left')
    df['is_repeat_customer'] = (df['customer_order_count'] > 1).astype(int)
    
    print(f'Prepared {len(df)} orders inline')

print(f'Delivery rate: {df["is_delivered"].mean():.3f}')
print(f'Columns: {list(df.columns)}')

## 2. Feature Engineering

In [ ]:
# Initialize feature engineer with historical data for regional/category stats
fe = FeatureEngineer(historical_data=df)

# Transform all orders
X_features = fe.transform_dataframe(df)
y = df['is_delivered'].values

feature_names = FeatureEngineer.get_feature_names()
X = X_features[feature_names]

print(f'Feature matrix shape: {X.shape}')
print(f'Target distribution: {pd.Series(y).value_counts().to_dict()}')
print(f'\nFeatures:')
for i, name in enumerate(feature_names):
    print(f'  {i+1}. {name}: min={X[name].min():.2f}, max={X[name].max():.2f}, mean={X[name].mean():.2f}')

## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples ({y_train.mean():.3f} positive rate)')
print(f'Test set:     {X_test.shape[0]} samples ({y_test.mean():.3f} positive rate)')

## 4. Train XGBoost Model

In [ ]:
# Calculate scale_pos_weight for class imbalance
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1

print(f'Class balance - Positive: {pos_count}, Negative: {neg_count}')
print(f'scale_pos_weight: {scale_pos_weight:.3f}')

# Train initial model
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42,
    use_label_encoder=False,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

print('Model trained!')

## 5. Model Evaluation

In [ ]:
# Predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Metrics
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Failed', 'Delivered']))

auc_score = roc_auc_score(y_test, y_proba)
print(f'AUC-ROC Score: {auc_score:.4f}')
print(f'Accuracy:      {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score:      {f1_score(y_test, y_pred):.4f}')

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Failed', 'Delivered'], yticklabels=['Failed', 'Delivered'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc_score:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_proba)
axes[2].plot(recall, precision, color='orange', lw=2)
axes[2].set_title('Precision-Recall Curve')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')

plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_names,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(importance['feature'], importance['importance'], color='steelblue')
plt.title('Feature Importance (XGBoost)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('\nTop 5 most important features:')
for _, row in importance.tail(5).iterrows():
    print(f'  {row["feature"]}: {row["importance"]:.4f}')

## 7. Hyperparameter Tuning (Optional)

In [ ]:
# Grid search for best hyperparameters
# This can take a few minutes — skip if time-constrained

RUN_GRID_SEARCH = False  # Set to True to run

if RUN_GRID_SEARCH:
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.05, 0.1, 0.15],
        'subsample': [0.7, 0.8, 0.9],
    }
    
    grid_model = XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric='auc',
        random_state=42,
        use_label_encoder=False,
    )
    
    grid_search = GridSearchCV(
        grid_model, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1
    )
    grid_search.fit(X_train, y_train)
    
    print(f'Best params: {grid_search.best_params_}')
    print(f'Best AUC-ROC: {grid_search.best_score_:.4f}')
    
    # Use the best model
    model = grid_search.best_estimator_
    y_proba = model.predict_proba(X_test)[:, 1]
    print(f'Test AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}')
else:
    print('Grid search skipped. Set RUN_GRID_SEARCH = True to run.')

## 8. Cross-Validation

In [ ]:
# 5-fold cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc', n_jobs=-1)
print(f'Cross-Validation AUC-ROC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print(f'Individual folds: {[f"{s:.4f}" for s in cv_scores]}')

## 9. Save the Trained Model

In [ ]:
# Save model and feature engineer
joblib.dump(model, MODEL_DIR / 'risk_model.joblib')
joblib.dump(fe, MODEL_DIR / 'feature_engineer.joblib')

print(f'Model saved to: {MODEL_DIR / "risk_model.joblib"}')
print(f'Feature engineer saved to: {MODEL_DIR / "feature_engineer.joblib"}')

# Verify loading
loaded_model = joblib.load(MODEL_DIR / 'risk_model.joblib')
loaded_fe = joblib.load(MODEL_DIR / 'feature_engineer.joblib')
test_proba = loaded_model.predict_proba(X_test[:5])[:, 1]
print(f'\nVerification - test predictions: {[f"{p:.3f}" for p in test_proba]}')
print('Model saved and verified successfully!')

## 10. Test Risk Prediction API Format

In [ ]:
from app.models.predictor import OrderRiskPredictor

# Test the predictor class
predictor = OrderRiskPredictor()
predictor.load()

# Simulate a high-risk order
risky_order = {
    'total_amount': 25000,
    'shipping_cost': 900,
    'subtotal': 24100,
    'discount': 0,
    'n_items': 3,
    'customer_phone_2': None,
    'customer_state': 'AC',  # Remote zone 3
    'product_category': 'electronics',
    'order_date': '2024-01-15 22:30:00',
    'is_repeat_customer': False,
    'customer_order_count': 0,
    'customer_success_rate': 0.0,
    'estimated_delivery_days': 15,
    'avg_product_weight': 3.5,
}

result = predictor.predict(risky_order)
print('=== High-Risk Order ===')
print(f'Score: {result["score"]}/100')
print(f'Category: {result["category"]}')
print(f'Recommendation: {result["recommendation"]}')
print(f'Reasons:')
for r in result['reasons']:
    print(f'  - {r}')

print()

# Simulate a low-risk order
safe_order = {
    'total_amount': 3500,
    'shipping_cost': 400,
    'subtotal': 3100,
    'discount': 0,
    'n_items': 1,
    'customer_phone_2': '0661234567',
    'customer_state': 'SP',  # Zone 1
    'product_category': 'health_beauty',
    'order_date': '2024-01-15 14:00:00',
    'is_repeat_customer': True,
    'customer_order_count': 5,
    'customer_success_rate': 0.9,
    'estimated_delivery_days': 5,
    'avg_product_weight': 0.5,
}

result = predictor.predict(safe_order)
print('=== Low-Risk Order ===')
print(f'Score: {result["score"]}/100')
print(f'Category: {result["category"]}')
print(f'Recommendation: {result["recommendation"]}')